# `code/pipeline/p001_45_ladder_pipeline.py`

Read-only rendering of the script (no outputs; it needs the licensed inputs described in `../../DATA_ACCESS.md`). The .py file is the version of record.


```text
p001_45 — P5: 금융 사다리 파이프라인 분해 — 후기단계에서 여성 파트너 채널이 좁아지는 마진은 어디인가

[왜] §6 의 사다리 결과(20.2% → 13.3%, 차등 기울기 +3.91, 재배분 반사실 +26.8%)는 "구성 산술"이다. 설계 제안 P5: 좁아짐은
 관측 가능한 마진의 곱이다 — (1) 여성창업 기업이 후기 라운드에 덜 도달(창업자 측 사실) (2) 도달해도 여성 파트너가 들여온 회사가
 덜 후속 참여 (3) 회사가 참여해도 같은 여성 파트너가 덜 재귀속 (4) 후기 라운드의 신규 진입 투자자 자체가 더 남성. (2)·(3)은
 파트너 귀속이 있어야 잴 수 있는 공급 측 사실이다. 전부 기술적(descriptive)이며 인과 언어를 쓰지 않는다.

[구성] 모집단 sample_v1 기업(창업자 성별 관측, NAEU). 지분형 라운드 시퀀스(EQ_EXCL 제외).
 (1) 첫 라운드가 초기(pre_seed/seed/angel/convertible)인 2010–2019 기업: series_a 도달, series_b+ 도달 — FF vs 기타, 첫 라운드
     연도×섹터×국가 셀 내 demean, 기업 군집 부트. 경쟁 위험판: 도달 전 인수·IPO 기업 제외.
 (2) 귀속 딜(≤2020-10) 중 36m 내 다음 라운드 有: y = 회사가 다음 라운드 투자자에 포함. cell_stage 내 fp + fp×ff → 이중차.
 (3) (2) 조건(회사 참여) 위에서 y = 같은 파트너 재귀속.
 (4) 다음 라운드의 신규 진입 투자자(그 기업 이전 라운드에 없던 회사) 중 귀속 有 행의 여성 파트너 비중, 다음 라운드 단계(초기/후기)×ff.
 커버리지 DiD(필수): 다음 라운드 투자자 기록 有(라운드 수준) · 회사 참여 조건 시 파트너 귀속 기록 有 ← fp×ff.
 조건화 사건 균형: P(36m 내 다음 라운드) ← fp + fp×ff (cell_stage 내).
 후기 FF 딜의 귀속 파트너 구성: 기존 관계자(같은 기업 이전 라운드에 귀속) vs 신규, fp 별.
[사전 예측] (2026-09-09, 결과 조회 전)
 (1) series_b+ 도달 FF−기타 ∈ [−0.07, −0.03] (셀 내), 경쟁 위험판도 음. (2) 이중차 ∈ [−0.06, 0], CI 0 포함(MDE ≈ 4pp). (3) ∈ [−0.06, 0].
 (4) 신규 진입 fp 비중: 후기 < 초기, FF·기타 모두. 조건화 균형 |fp×ff| < 0.02. 커버리지 DiD ≈ 0 (|b| < 0.02).
[판정] 기술 통계 — status OK. 마진 (2)/(3) 이중차 CI 0 배제 시 verdict 에 표기.
```


In [ ]:
import numpy as np
import pandas as pd

from p001_v6_common import (CUT, EARLY, LATE, RESCUE_SHA, V6_SHA, boot, emit, equity_rounds, exit_dates, fit, investor_rows, load_sample, log,
                            org_maps, partner_gender_rows, qci)

rng = np.random.default_rng(20260945)
NB = 300
OUT = {}
d = load_sample()
om = org_maps(d)
R = equity_rounds(set(d["org_uuid"]))
R["ff"] = R["org_uuid"].map(om["ff"]); R["cat"] = R["org_uuid"].map(om["cat"])
ex = exit_dates()


In [ ]:
# ── (1) 기업 소모 ──────────────────────────────────────────────────────────
log("=" * 100 + "\n[1] 기업 소모: 초기 첫 라운드(2010–2019) 기업의 series_a / series_b+ 도달\n" + "=" * 100)
first = R[R["seq"] == 0].copy()
first = first[first["investment_type"].isin(EARLY) & (first["rdt"] >= "2010-01-01") & (first["rdt"] <= "2019-12-31")]
reach_a = R[R["investment_type"] == "series_a"].groupby("org_uuid")["rdt"].min()
reach_b = R[R["investment_type"].isin(LATE)].groupby("org_uuid")["rdt"].min()
C1 = first[["org_uuid", "rdt", "ff", "cat"]].copy()
C1["country"] = C1["org_uuid"].map(d.groupby("org_uuid")["country_code"].first())
C1["cell"] = C1["rdt"].dt.year.astype(str) + "|" + C1["cat"].astype(str) + "|" + C1["country"].astype(str)
C1["reach_a"] = C1["org_uuid"].map(reach_a).notna().astype(float)
C1["reach_b"] = C1["org_uuid"].map(reach_b).notna().astype(float)
C1["exit_dt"] = C1["org_uuid"].map(ex)
C1["exit_before_b"] = (C1["exit_dt"].notna() & (C1["exit_dt"] < C1["org_uuid"].map(reach_b).fillna(pd.Timestamp("2100-01-01")))).astype(float)
C1["firm"] = C1["org_uuid"]  # boot 군집 열 규약
M1 = {"n_companies": int(len(C1)), "n_ff": int(C1["ff"].sum()), "raw_reach_a": {"ff": round(float(C1.loc[C1.ff == 1, "reach_a"].mean()), 4), "other": round(float(C1.loc[C1.ff == 0, "reach_a"].mean()), 4)},
      "raw_reach_b": {"ff": round(float(C1.loc[C1.ff == 1, "reach_b"].mean()), 4), "other": round(float(C1.loc[C1.ff == 0, "reach_b"].mean()), 4)}}
M1["reach_a_cell"] = boot(C1, "reach_a", ["ff"], ["ff"], rng, nb=NB, demean="cell", cluster="firm")
M1["reach_b_cell"] = boot(C1, "reach_b", ["ff"], ["ff"], rng, nb=NB, demean="cell", cluster="firm")
C1x = C1[C1["exit_before_b"] == 0]
M1["reach_b_cell_excl_exit"] = boot(C1x, "reach_b", ["ff"], ["ff"], rng, nb=NB, demean="cell", cluster="firm")
M1["share_exit_before_b"] = {"ff": round(float(C1.loc[C1.ff == 1, "exit_before_b"].mean()), 4), "other": round(float(C1.loc[C1.ff == 0, "exit_before_b"].mean()), 4)}
log(f"  기업 {M1['n_companies']:,} (FF {M1['n_ff']:,}) · series_a 도달 FF {M1['raw_reach_a']['ff']:.3f} vs {M1['raw_reach_a']['other']:.3f} · series_b+ {M1['raw_reach_b']['ff']:.3f} vs {M1['raw_reach_b']['other']:.3f}")
for k in ("reach_a_cell", "reach_b_cell", "reach_b_cell_excl_exit"):
    r = M1[k]; log(f"  {k:<24} ff {r['ff']['coef']:+.4f} [{r['ff']['ci95'][0]:+.4f},{r['ff']['ci95'][1]:+.4f}] n={r['n']:,}")
OUT["M1_attrition"] = M1


In [ ]:
# ── (2)/(3) 귀속 딜의 회사 후속 참여·파트너 재귀속 ──────────────────────────
log("\n" + "=" * 100 + "\n[2]/[3] 회사 후속 참여 · 같은 파트너 재귀속 (cell_stage 내, fp + fp×ff)\n" + "=" * 100)
rr = R.set_index("uuid")
A = d[d["dt"] <= "2020-10-31"].copy()
A["next_uuid"] = A["funding_round_uuid"].map(rr["next_uuid"]); A["next_dt"] = A["funding_round_uuid"].map(rr["next_dt"]); A["next_type"] = A["funding_round_uuid"].map(rr["next_type"])
A["next36"] = ((A["next_dt"] - A["dt"]).dt.days <= 1095).fillna(False).astype(float)
A = A[A["next_uuid"].notna() | True].copy()
inv_next = investor_rows(set(A["next_uuid"].dropna()))
next_inv = inv_next.groupby("funding_round_uuid")["investor_uuid"].agg(set).to_dict()
next_has_inv = {k: 1.0 for k in next_inv}
pt_next = partner_gender_rows(set(A["next_uuid"].dropna()))
next_part = pt_next.groupby(["funding_round_uuid", "investor_uuid"])["partner_uuid"].agg(frozenset).to_dict()
next_any_part = pt_next.groupby(["funding_round_uuid", "investor_uuid"]).size().to_dict()
A["fpff"] = A["fp"] * A["ff"]
A["firm"] = A["investor_uuid"]
Ac = A[A["next36"] == 1].copy()
Ac["y_firm"] = [1.0 if (inv in next_inv.get(nu, set())) else 0.0 for nu, inv in zip(Ac["next_uuid"], Ac["investor_uuid"])]
Ac["rec_next_inv"] = [1.0 if nu in next_has_inv else 0.0 for nu in Ac["next_uuid"]]
Ac["y_partner"] = [1.0 if p in next_part.get((nu, inv), frozenset()) else 0.0 for nu, inv, p in zip(Ac["next_uuid"], Ac["investor_uuid"], Ac["partner_uuid"])]
Ac["rec_next_part"] = [1.0 if (nu, inv) in next_any_part else 0.0 for nu, inv in zip(Ac["next_uuid"], Ac["investor_uuid"])]
cells = Ac.groupby(["fp", "ff"]).agg(y_firm=("y_firm", "mean"), n=("y_firm", "size"))
M2 = {"n_deals_cond": int(len(Ac)), "four_cell_firm_followon": {f"fp{int(a)}_ff{int(b)}": round(float(v), 4) for (a, b), v in cells["y_firm"].items()}}
M2["cond_balance"] = boot(A, "next36", ["fp", "fpff", "ff"], ["fp", "fpff"], rng, nb=NB, demean="cell_stage", cluster="firm")
M2["firm_followon"] = boot(Ac, "y_firm", ["fp", "fpff", "ff"], ["fp", "fpff"], rng, nb=NB, demean="cell_stage", cluster="firm")
M2["coverage_next_investors"] = boot(Ac, "rec_next_inv", ["fp", "fpff", "ff"], ["fp", "fpff"], rng, nb=NB, demean="cell_stage", cluster="firm")
Af = Ac[Ac["y_firm"] == 1].copy()
cells3 = Af.groupby(["fp", "ff"]).agg(y=("y_partner", "mean"), n=("y_partner", "size"))
M3 = {"n_deals_cond_firm": int(len(Af)), "four_cell_repartner": {f"fp{int(a)}_ff{int(b)}": round(float(v), 4) for (a, b), v in cells3["y"].items()}}
M3["partner_reattr"] = boot(Af, "y_partner", ["fp", "fpff", "ff"], ["fp", "fpff"], rng, nb=NB, demean="cell_stage", cluster="firm")
M3["coverage_next_partner"] = boot(Af, "rec_next_part", ["fp", "fpff", "ff"], ["fp", "fpff"], rng, nb=NB, demean="cell_stage", cluster="firm")
for tag, r in (("조건화 균형 next36", M2["cond_balance"]), ("회사 후속 참여", M2["firm_followon"]), ("커버리지: 다음 라운드 투자자 기록", M2["coverage_next_investors"]),
               ("파트너 재귀속|회사 참여", M3["partner_reattr"]), ("커버리지: 파트너 귀속 기록", M3["coverage_next_partner"])):
    log(f"  {tag:<28} fp {r['fp']['coef']:+.4f} [{r['fp']['ci95'][0]:+.3f},{r['fp']['ci95'][1]:+.3f}] · fp×ff {r['fpff']['coef']:+.4f} [{r['fpff']['ci95'][0]:+.3f},{r['fpff']['ci95'][1]:+.3f}] MDE {r['fpff']['mde80']:.3f} n={r['n']:,}")
log(f"  4셀 회사 후속 {M2['four_cell_firm_followon']} · 4셀 재귀속 {M3['four_cell_repartner']}")
OUT["M2_firm_followon"] = M2; OUT["M3_partner_reattribution"] = M3


In [ ]:
# ── (4) 신규 진입 투자자의 여성 파트너 비중 ─────────────────────────────────
log("\n" + "=" * 100 + "\n[4] 다음 라운드 신규 진입 투자자(귀속 有)의 여성 파트너 비중: 단계 × ff\n" + "=" * 100)
prev_inv_sets = {}
inv_all = investor_rows(set(R["uuid"]))
r_inv = inv_all.groupby("funding_round_uuid")["investor_uuid"].agg(set).to_dict()
Rs = R.sort_values(["org_uuid", "rdt"])
seen = {}
cum = []
for org, ru in zip(Rs["org_uuid"], Rs["uuid"]):
    s = seen.get(org, set()); cum.append(frozenset(s)); seen[org] = s | r_inv.get(ru, set())
Rs["prev_investors"] = cum
pt_all = partner_gender_rows(set(R["uuid"]))
pa = pt_all.groupby(["funding_round_uuid", "investor_uuid"])["fp"].max().reset_index()
pa = pa.merge(Rs[["uuid", "org_uuid", "rdt", "investment_type", "ff", "prev_investors", "seq"]], left_on="funding_round_uuid", right_on="uuid")
pa = pa[(pa["seq"] >= 1) & (pa["rdt"] >= "2010-01-01") & (pa["rdt"] <= CUT)]
pa["entrant"] = [inv not in prev for inv, prev in zip(pa["investor_uuid"], pa["prev_investors"])]
pa["late"] = pa["investment_type"].isin(LATE).astype(float)
ent = pa[pa["entrant"]]
M4 = {"n_entrant_rows": int(len(ent)), "fp_share_entrants": {f"late{int(l)}_ff{int(f)}": {"share": round(float(g["fp"].mean()), 4), "n": int(len(g))} for (l, f), g in ent.groupby(["late", "ff"])},
      "fp_share_incumbents": {f"late{int(l)}_ff{int(f)}": {"share": round(float(g["fp"].mean()), 4), "n": int(len(g))} for (l, f), g in pa[~pa["entrant"]].groupby(["late", "ff"])}}
ent = ent.assign(firm=ent["org_uuid"], lateff=ent["late"] * ent["ff"])
M4["entrant_fp_on_late"] = boot(ent, "fp", ["late", "lateff", "ff"], ["late", "lateff"], rng, nb=NB, cluster="firm")
log(f"  신규 진입 fp 비중 {M4['fp_share_entrants']} · 기존 관계자 {M4['fp_share_incumbents']}")
r = M4["entrant_fp_on_late"]; log(f"  fp ← late {r['late']['coef']:+.4f} [{r['late']['ci95'][0]:+.3f},{r['late']['ci95'][1]:+.3f}] · late×ff {r['lateff']['coef']:+.4f} [{r['lateff']['ci95'][0]:+.3f},{r['lateff']['ci95'][1]:+.3f}]")
OUT["M4_entrants"] = M4
# 후기 FF 딜의 귀속 파트너: 기존 관계자 vs 신규, fp 별
lateff = pa[(pa["late"] == 1) & (pa["ff"] == 1)]
OUT["M5_late_ff_incumbent_share"] = {"fp1": round(float(1 - lateff.loc[lateff.fp == 1, "entrant"].mean()), 4), "fp0": round(float(1 - lateff.loc[lateff.fp == 0, "entrant"].mean()), 4),
                                     "n_fp1": int((lateff.fp == 1).sum()), "n_fp0": int((lateff.fp == 0).sum())}
log(f"  후기 FF 딜 중 기존 관계자 비중: 여성 파트너 {OUT['M5_late_ff_incumbent_share']['fp1']:.3f} vs 남성 {OUT['M5_late_ff_incumbent_share']['fp0']:.3f}")


In [ ]:
# ── 판정 ────────────────────────────────────────────────────────────────────
b2, b3 = M2["firm_followon"]["fpff"], M3["partner_reattr"]["fpff"]
pred = {"M1_reach_b_in_[-0.07,-0.03]": -0.07 <= M1["reach_b_cell"]["ff"]["coef"] <= -0.03, "M1_excl_exit_neg": M1["reach_b_cell_excl_exit"]["ff"]["coef"] < 0,
        "M2_dd_in_[-0.06,0]_incl0": -0.06 <= b2["coef"] <= 0 and not b2["sig"], "M3_dd_in_[-0.06,0]": -0.06 <= b3["coef"] <= 0,
        "M4_entrant_fp_lower_late": M4["entrant_fp_on_late"]["late"]["coef"] < 0, "cond_balance_lt_0.02": abs(M2["cond_balance"]["fpff"]["coef"]) < 0.02,
        "coverage_dd_lt_0.02": abs(M2["coverage_next_investors"]["fpff"]["coef"]) < 0.02 and abs(M3["coverage_next_partner"]["fpff"]["coef"]) < 0.02}
pred = {k: bool(v) for k, v in pred.items()}
OUT["prediction_check"] = pred
verdict = (f"(1) series_b+ 도달 FF−기타 셀 내 {M1['reach_b_cell']['ff']['coef']:+.4f} [{M1['reach_b_cell']['ff']['ci95'][0]:+.3f},{M1['reach_b_cell']['ff']['ci95'][1]:+.3f}] (출구 제외 {M1['reach_b_cell_excl_exit']['ff']['coef']:+.4f}) | "
           f"(2) 회사 후속 fp×ff {b2['coef']:+.4f} [{b2['ci95'][0]:+.3f},{b2['ci95'][1]:+.3f}] MDE {b2['mde80']:.3f} | (3) 재귀속 fp×ff {b3['coef']:+.4f} [{b3['ci95'][0]:+.3f},{b3['ci95'][1]:+.3f}] | "
           f"(4) 신규 진입 fp ← late {M4['entrant_fp_on_late']['late']['coef']:+.4f} · late×ff {M4['entrant_fp_on_late']['lateff']['coef']:+.4f} | 조건화 균형 fp×ff {M2['cond_balance']['fpff']['coef']:+.4f} · "
           f"커버리지 DD {M2['coverage_next_investors']['fpff']['coef']:+.4f}/{M3['coverage_next_partner']['fpff']['coef']:+.4f} — "
           f"{'이중차 검출: ' + ', '.join(k for k, b in (('회사 후속', b2), ('재귀속', b3)) if b['sig']) if (b2['sig'] or b3['sig']) else '이중차 미검출(MDE 병기)'} (예측 적중 {sum(pred.values())}/{len(pred)})")
emit("P001-45", "P5: 금융 사다리 파이프라인 — 기업 소모 · 회사 후속 참여 · 파트너 재귀속 · 신규 진입 구성 (기술 통계)", "OK", OUT,
     prediction="series_b+ FF−기타 ∈[−0.07,−0.03]; 이중차 (2)(3) ∈[−0.06,0] CI 0 포함; 신규 진입 fp 후기<초기; 조건화 균형·커버리지 DD |b|<0.02",
     verdict=verdict, kill_met=False, n=int(M2["n_deals_cond"]),
     extra={"stage": 7, "feeds": "v6 설계 제안 P5 → §6 사다리 문단", "slug": "ladder_pipeline", "builds_on": "P001-06/17", "common_sha256_16": RESCUE_SHA, "v6_common_sha256_16": V6_SHA})
log("done")
